# 04 · Türkçe talepten sipariş taslağına (Gemma akışı)

`core.llm.conversation` Türkçe talebi sipariş taslağına çevirir, eksik bilgiyi sorar ve **ayrı insan onayı**
olmadan simülasyon çalıştırmaz.

> **Önemli:** Bu notebook gerçek Gemma'yı çağırmaz. Model yanıtları aşağıdaki `ScriptedClient` ile
> **elle yazılmış test yanıtlarıdır** (repodaki birim testleriyle aynı yöntem). Amaç akışı ve güvenlik
> kurallarını göstermek; model doğruluğu iddiası değildir. Gerçek model için README'deki Ollama komutlarına bakın.

In [1]:
# Notebook'u tutorials/ veya repo kökünden açabilirsiniz; importlar repo kökünden çalışır.
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = "notebook_connected"   # plotly.js'i her notebook'a gömmez; görüntülemek için internet gerekir
print("Repo kökü:", ROOT)

Repo kökü: /Users/nazlsu/Desktop/PickWise


In [2]:
from benchmarks.workshop_order_demo import DemoInput
from core.llm.conversation import IntentDecision, OrderFields, answer_intake, begin_intake
from core.llm.intake import confirm_and_simulate
from core.scenarios import create_snapshot

demo = DemoInput.model_validate_json(Path("data/examples/order_demo.json").read_text(encoding="utf-8"))
baseline = create_snapshot(demo.baseline, "v1", time_limit=5)


class ScriptedClient:
    """TEST İSTEMCİSİ: gerçek model değil. Önceden yazılmış alanları döndürür."""

    def __init__(self, fields):
        self.fields = fields

    def structured(self, schema, system, data, prompt_version):
        if schema is IntentDecision:
            value = IntentDecision(tool="simulate_order", evidence="Yeni sipariş")
        else:
            value = OrderFields.model_validate(self.fields)
        return value, {"test_double": True, "prompt_version": prompt_version}


def fields(machine="Torna 1", duration="10 dakika", **extra):
    base = dict(order_name=None,
                operations=[{"name": "Tornalama", "options": [{"machine_text": machine, "duration_text": duration}]}],
                deadline_kind="none", deadline_text=None, preservation="none", preservation_text=None)
    base.update(extra)
    return base

## 1. Eksik süre → soru

Kullanıcı süreyi söylemedi. Sistem tahmin etmez, sorar.

In [3]:
session = begin_intake("Yeni sipariş: Torna 1 üzerinde tornalama yapacağız.", baseline,
                       ScriptedClient(fields(duration=None)))
print("Durum:", session.result.state)
for question in session.result.questions:
    print("Soru:", question)

Durum: clarification
Soru: 1. işlemin bu makinedeki süresi nedir? Örnek: 25 dakika veya iki saat.


In [4]:
session = answer_intake(session, "iki saat", baseline)
print("Durum:", session.result.state)
print("Taslak süreler:", [op.processing_times for op in session.result.proposed_order.operations])

Durum: confirmation_required
Taslak süreler: [{100: 120}]


Yanıt kodla dakikaya çevrildi (`iki saat` → 120). Diğer alanlar yeniden çıkarılmadı.

## 2. Belirsiz makine → seçenek

In [5]:
ambiguous = begin_intake("Yeni sipariş torna üzerinde 10 dakika.", baseline, ScriptedClient(fields(machine="torna")))
print("Bekleyen alan:", ambiguous.pending[0].kind, "| seçenekler:", ambiguous.pending[0].choices)
ambiguous = answer_intake(ambiguous, "Torna 2", baseline)
print("Seçilen:", ambiguous.result.proposed_order.operations[0].processing_times)

Bekleyen alan: machine | seçenekler: ['Torna 1', 'Torna 2']
Seçilen: {200: 10}


## 3. Onay olmadan simülasyon yok

In [6]:
try:
    confirm_and_simulate(session.result, baseline, "talep_sim_1", user_confirmed=False)
except ValueError as error:
    print("Durduruldu:", error)

simulation = confirm_and_simulate(session.result, baseline, "talep_sim_1", user_confirmed=True)
print("Onaydan sonra:", simulation.result.status, "| makespan",
      simulation.comparison.before.makespan, "→", simulation.comparison.after.makespan, "dk")

Durduruldu: A complete proposed order and explicit human confirmation are required
Onaydan sonra: OPTIMAL | makespan 15 → 135 dk


## 4. Gerçek model var mı?

Yerelde Ollama çalışıyorsa aşağıdaki hücre model bilgisini gösterir; yoksa uygulama **manuel girişle** çalışmaya devam eder.

In [7]:
from core.llm.ollama_client import ModelUnavailable, OllamaClient

try:
    print(OllamaClient(timeout=3).metadata())
except ModelUnavailable as error:
    print("Gemma şu an erişilemiyor; manuel giriş yolu kullanılır.", error)

Gemma şu an erişilemiyor; manuel giriş yolu kullanılır. Model is not installed: gemma4:e2b
